In [ ]:
%pip install undetected-chromedriver
%pip install selenium

In [7]:
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
import pandas as pd
import time
import random

# 0) 동명이인 시 선택할 ‘최근 소속팀’ 입력
selected_team = "한화"

# 1) 크롤링 대상 선수 & 연도
players      = ["조동욱", "한승혁", "황준서", "정우주", "주현상", "김범수", "김종수", "김기중", "김서현", "박상원"]
target_years = ['2025']

# 2) 브라우저 실행 및 초기 화면
driver  = uc.Chrome(version_main=138)
actions = ActionChains(driver)
driver.get("https://statiz.sporki.com/")
time.sleep(random.uniform(2.0, 3.0))

for player_name in players:
    all_data = []
    print(f"\n🎯 {player_name} 크롤링 시작...")

    # 검색 창 열기
    search_button = driver.find_element(By.CSS_SELECTOR, "a.btn_box")
    actions.move_to_element(search_button).click().perform()
    time.sleep(random.uniform(1.0, 1.5))

    # 이름 입력 후 엔터
    search_input = driver.find_element(By.ID, "s")
    search_input.clear()
    search_input.send_keys(player_name)
    time.sleep(random.uniform(0.5, 1.0))
    search_input.send_keys(Keys.RETURN)
    time.sleep(random.uniform(2.0, 3.0))

    # 동명이인 페이지 감지
    duplicates = driver.find_elements(By.CSS_SELECTOR, "div.notice_txt")
    if duplicates:
        print(f"⚠️ 동명이인: '{player_name}' – '{selected_team}' & P 포지션으로 필터")
        table   = driver.find_element(By.CSS_SELECTOR, "table")
        headers = [th.text.strip() for th in table.find_elements(By.CSS_SELECTOR, "thead th")]
        idx_team = headers.index("최근 소속팀")
        idx_pos  = headers.index("최근 주포지션")

        found = False
        for tr in table.find_elements(By.CSS_SELECTOR, "tbody tr"):
            tds = tr.find_elements(By.TAG_NAME, "td")
            if (tds[idx_team].text.strip() == selected_team and
                tds[idx_pos].text.strip() == "P"):
                link = tds[0].find_element(By.TAG_NAME, "a")
                actions.move_to_element(link).click().perform()
                found = True
                break

        if not found:
            print(f"❌ {player_name}: 투수가 아닙니다. 스킵합니다.")
            continue
    else:
        # 일반 검색 결과가 이미 선수 상세페이지라면 패스
        pass

    time.sleep(random.uniform(2.0, 3.0))

    # 상세분석 탭 클릭
    analysis_tab = driver.find_element(By.CLASS_NAME, "p_analytics")
    actions.move_to_element(analysis_tab).click().perform()
    time.sleep(random.uniform(2.0, 3.0))

    # 평균 구속 옵션 선택
    si2_btn = driver.find_element(By.CSS_SELECTOR, "#select_si2 .label")
    actions.move_to_element(si2_btn).click().perform()
    time.sleep(random.uniform(0.5, 1.0))
    for opt in driver.find_elements(By.CSS_SELECTOR, "#select_si2 .option_item"):
        if opt.get_attribute("value") == "3":
            actions.move_to_element(opt).click().perform()
            break
    time.sleep(random.uniform(2.0, 3.0))

    # 연도별 데이터 수집
    for year in target_years:
        print(f"📅 {player_name} - {year}년 크롤링 중...")

        # 연도 드롭다운 열기
        #year_btn = driver.find_element(By.CSS_SELECTOR, "#select_year .label")
        #actions.move_to_element(year_btn).click().perform()
        #time.sleep(random.uniform(0.8, 1.2))

        # 옵션 확인
        #options = driver.find_elements(By.CSS_SELECTOR, "#select_year .option_item")
        #values  = [opt.get_attribute("value") for opt in options]
        #if year not in values:
            #print(f"⚠️ {player_name} - {year}년 버튼 없음")
            #actions.move_to_element(year_btn).click().perform()
            #time.sleep(0.5)
            #continue

        # 해당 연도 선택
        #for opt in options:
            #if opt.get_attribute("value") == year:
                #actions.move_to_element(opt).click().perform()
                #break
        #time.sleep(random.uniform(2.0, 3.0))
        
        # 연도 드롭다운 열고 '2025' 직접 클릭
        try:
            year_btn = driver.find_element(By.CSS_SELECTOR, "#select_year .label")
            actions.move_to_element(year_btn).click().perform()
            time.sleep(random.uniform(0.5, 1.0))

            opt_2025 = driver.find_element(By.CSS_SELECTOR, "#select_year .option_item[value='2025']")
            actions.move_to_element(opt_2025).click().perform()
            time.sleep(random.uniform(2.0, 3.0))
        except Exception as e:
            print(f"⚠️ {player_name} - 2025년 연도 선택 실패: {e}")
            continue

        # 테이블 수집
        rows = driver.find_elements(By.CSS_SELECTOR, ".table_type03 table tr")[2:]
        if not rows:
            print(f"⚠️ {player_name} - {year}년 데이터 없음")
            continue

        for row in rows:
            cols = row.find_elements(By.TAG_NAME, "td")
            values = [player_name, year] + [c.text.strip() for c in cols[:13]]
            all_data.append(values)
        time.sleep(random.uniform(1.0, 2.0))

    # 데이터프레임 생성 및 날짜 결측값 제거
    columns = [
        '선수명','연도','날짜','상대팀','전체구속','2Seam','4Seam','Cutter',
        'Curve','Slider','Changeup','Sinker','Forkball','Knuckle','Other'
    ]
    df = pd.DataFrame(all_data, columns=columns)
    df = df.dropna(subset=['날짜'])

    # CSV 저장
    filename = f"{player_name}_평균구속.csv"
    df.to_csv(filename, index=False, encoding="utf-8-sig")
    print(f"✅ 저장 완료: {filename}")

driver.quit()
print("\n🎉 모든 선수 크롤링 완료!")


🎯 조동욱 크롤링 시작...
📅 조동욱 - 2025년 크롤링 중...
✅ 저장 완료: 조동욱_평균구속.csv

🎯 한승혁 크롤링 시작...
⚠️ 동명이인: '한승혁' – '한화' & P 포지션으로 필터
📅 한승혁 - 2025년 크롤링 중...
✅ 저장 완료: 한승혁_평균구속.csv

🎯 황준서 크롤링 시작...
📅 황준서 - 2025년 크롤링 중...
✅ 저장 완료: 황준서_평균구속.csv

🎯 정우주 크롤링 시작...
📅 정우주 - 2025년 크롤링 중...
✅ 저장 완료: 정우주_평균구속.csv

🎯 주현상 크롤링 시작...
📅 주현상 - 2025년 크롤링 중...
✅ 저장 완료: 주현상_평균구속.csv

🎯 김범수 크롤링 시작...
📅 김범수 - 2025년 크롤링 중...
✅ 저장 완료: 김범수_평균구속.csv

🎯 김종수 크롤링 시작...
⚠️ 동명이인: '김종수' – '한화' & P 포지션으로 필터
📅 김종수 - 2025년 크롤링 중...
✅ 저장 완료: 김종수_평균구속.csv

🎯 김기중 크롤링 시작...
📅 김기중 - 2025년 크롤링 중...
✅ 저장 완료: 김기중_평균구속.csv

🎯 김서현 크롤링 시작...
📅 김서현 - 2025년 크롤링 중...
✅ 저장 완료: 김서현_평균구속.csv

🎯 박상원 크롤링 시작...
⚠️ 동명이인: '박상원' – '한화' & P 포지션으로 필터
📅 박상원 - 2025년 크롤링 중...
✅ 저장 완료: 박상원_평균구속.csv

🎉 모든 선수 크롤링 완료!


In [18]:
import os
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time, random

# ── 저장할 디렉토리 생성 ──
output_dir = "구사율"
os.makedirs(output_dir, exist_ok=True)

selected_team = "SSG"
players = [
    "곽빈", "어빈", "김택연", "이영하", "로그", "박치국", "박정수", "박신지"
]
target_years  = ['2025']
pitch_types   = ["투심","포심","커터","커브","슬라","첸접","싱커","포크","너클","기타"]

# ── Chrome 옵션에 user-agent 추가 ──
options = uc.ChromeOptions()
options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/114.0.0.0 Safari/537.36"
)

driver  = uc.Chrome(options=options, version_main=138)
actions = ActionChains(driver)
wait    = WebDriverWait(driver, 10)
all_rows = []

for player_name in players:
    driver.get("https://statiz.sporki.com/")
    time.sleep(random.uniform(2, 4))

    # ── 검색 ──
    actions.move_to_element(driver.find_element(By.CSS_SELECTOR, "a.btn_box")).click().perform()
    time.sleep(random.uniform(1, 2))

    # 검색창이 클릭 가능해질 때까지 대기하고, JS로 초기화
    search_input = wait.until(EC.element_to_be_clickable((By.ID, "s")))
    driver.execute_script("arguments[0].value = '';", search_input)

    # 선수명 입력 및 검색 실행
    search_input.send_keys(player_name)
    time.sleep(random.uniform(0.5, 1.0))
    search_input.send_keys(Keys.RETURN)
    time.sleep(random.uniform(2, 4))

    # ── 동명이인 처리 ──
    duplicates = driver.find_elements(By.CSS_SELECTOR, "div.notice_txt")
    if duplicates:
        print(f"⚠️ 동명이인 감지: {player_name} – '{selected_team}', P 포지션 필터링")
        table   = driver.find_element(By.CSS_SELECTOR, "table")
        headers = [th.text.strip() for th in table.find_elements(By.CSS_SELECTOR, "thead th")]
        idx_team = headers.index("최근 소속팀")
        idx_pos  = headers.index("최근 주포지션")

        found = False
        for tr in table.find_elements(By.CSS_SELECTOR, "tbody tr"):
            tds = tr.find_elements(By.TAG_NAME, "td")
            if (tds[idx_team].text.strip() == selected_team and
                tds[idx_pos].text.strip() == "P"):
                link = tds[0].find_element(By.TAG_NAME, "a")
                actions.move_to_element(link).click().perform()
                found = True
                break

        if not found:
            print(f"❌ {player_name}: 해당 팀의 투수가 아님, 스킵합니다.")
            time.sleep(random.uniform(1,2))
            continue
        time.sleep(random.uniform(2,3))

    # ── 연도별 → 구종 탭 ──
    driver.find_element(By.LINK_TEXT, "연도별").click()
    time.sleep(random.uniform(1, 2))
    driver.find_element(By.LINK_TEXT, "구종").click()
    time.sleep(random.uniform(1, 2))

    # ── 테이블 헤더 읽기 ──
    table = driver.find_element(By.CSS_SELECTOR, ".table_type02 table")
    header_ths = table.find_elements(By.CSS_SELECTOR, "thead tr:nth-child(1) th")
    spans = [int(th.get_attribute("colspan") or 1) for th in header_ths]

    # ① 구사율 블록 위치 찾기
    cum = 0
    for th, span in zip(header_ths, spans):
        if "구사율" in (th.get_attribute("tooltip") or th.text):
            pct_start, pct_count = cum, span
            break
        cum += span

    # ② 피안타율 블록 위치 찾기
    cum = 0
    for th, span in zip(header_ths, spans):
        tip = th.get_attribute("tooltip") or th.text
        if "피안타율" in tip and "피장타율" not in tip:
            avg_start, avg_count = cum, span
            break
        cum += span

    # ── tbody 순회하며 값 추출 ──
    for tr in table.find_elements(By.CSS_SELECTOR, "tbody tr"):
        tds = tr.find_elements(By.TAG_NAME, "td")
        year = tds[0].text.strip()
        if year in target_years:
            pct_vals = [
                float(td.get_attribute("innerText").strip().rstrip('%') or 0.0)
                for td in tds[pct_start:pct_start+pct_count]
            ]
            avg_vals = [
                float(td.get_attribute("innerText").strip().rstrip('%') or 0.0)
                for td in tds[avg_start:avg_start+avg_count]
            ]
            all_rows.append([player_name, int(year)] + pct_vals + avg_vals)
            print(f"▶ {player_name} {year} 구사율={pct_vals}, 피안타율={avg_vals}")

    # ── 선수마다 짧게 쉬기 ──
    time.sleep(random.uniform(5, 10))

driver.quit()

# ── DataFrame 생성 및 저장 ──
rate_cols = [f"{p}_구사율" for p in pitch_types]
avg_cols  = [f"{p}_피안타율" for p in pitch_types]
columns   = ["선수명","연도"] + rate_cols + avg_cols

df = pd.DataFrame(all_rows, columns=columns)
filepath = os.path.join(output_dir, "구사율_피안타율.csv")
df.to_csv(filepath, index=False, encoding="utf-8-sig")
print(f"✅ 완료: {filepath}")

▶ 곽빈 2025 구사율=[0.0, 45.0, 0.0, 14.6, 24.7, 15.3, 0.0, 0.0, 0.0, 0.0], 피안타율=[0.0, 0.316, 0.0, 0.219, 0.235, 0.111, 0.0, 0.0, 0.0, 0.0]
▶ 어빈 2025 구사율=[22.8, 35.0, 9.0, 16.7, 3.5, 12.6, 0.0, 0.0, 0.0, 0.0], 피안타율=[0.368, 0.236, 0.5, 0.211, 0.2, 0.213, 0.0, 0.0, 0.0, 0.0]
▶ 김택연 2025 구사율=[0.0, 73.4, 0.0, 0.5, 23.5, 0.0, 0.0, 2.4, 0.0, 0.0], 피안타율=[0.0, 0.162, 0.0, 0.0, 0.179, 0.0, 0.0, 0.667, 0.0, 0.0]
▶ 이영하 2025 구사율=[0.0, 49.3, 0.0, 7.6, 42.0, 0.0, 0.0, 1.0, 0.0, 0.0], 피안타율=[0.0, 0.279, 0.0, 0.143, 0.188, 0.0, 0.0, 1.0, 0.0, 0.0]
▶ 로그 2025 구사율=[25.6, 27.4, 13.9, 0.0, 20.6, 12.2, 0.0, 0.0, 0.0, 0.0], 피안타율=[0.233, 0.2, 0.295, 0.0, 0.115, 0.303, 0.0, 0.0, 0.0, 0.0]
▶ 박치국 2025 구사율=[28.3, 39.9, 0.0, 0.0, 28.2, 2.4, 0.0, 0.4, 0.0, 0.0], 피안타율=[0.306, 0.224, 0.0, 0.0, 0.217, 0.2, 0.0, 0.0, 0.0, 0.0]
▶ 박정수 2025 구사율=[3.4, 31.7, 0.0, 15.4, 28.0, 20.8, 0.0, 0.0, 0.0, 0.0], 피안타율=[0.0, 0.421, 0.0, 0.333, 0.4, 0.278, 0.0, 0.0, 0.0, 0.0]


NoSuchElementException: Message: no such element: Unable to locate element: {"method":"css selector","selector":".table_type02 table"}
  (Session info: chrome=138.0.7204.100); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#no-such-element-exception
Stacktrace:
	GetHandleVerifier [0x0xba44a3+62419]
	GetHandleVerifier [0x0xba44e4+62484]
	(No symbol) [0x0x9e2133]
	(No symbol) [0x0xa2a8fe]
	(No symbol) [0x0xa2ac9b]
	(No symbol) [0x0xa73052]
	(No symbol) [0x0xa4f4b4]
	(No symbol) [0x0xa7087a]
	(No symbol) [0x0xa4f266]
	(No symbol) [0x0xa1e852]
	(No symbol) [0x0xa1f6f4]
	GetHandleVerifier [0x0xe14793+2619075]
	GetHandleVerifier [0x0xe0fbaa+2599642]
	GetHandleVerifier [0x0xbcb04a+221050]
	GetHandleVerifier [0x0xbbb2c8+156152]
	GetHandleVerifier [0x0xbc1c7d+183213]
	GetHandleVerifier [0x0xbac388+94904]
	GetHandleVerifier [0x0xbac512+95298]
	GetHandleVerifier [0x0xb9766a+9626]
	BaseThreadInitThunk [0x0x74e75d49+25]
	RtlInit

In [5]:
import pandas as pd
from pathlib import Path

# —— 1) 파일 읽기 ——
# (이 부분을 실제 CSV 경로로 수정)
input_path = Path(r"C:\Users\yun72_92xubzr\OneDrive\바탕 화면\학술제\merged_최종.csv")
df = pd.read_csv(input_path, encoding='utf-8-sig', parse_dates=['Date'])

# —— 2) Appearances(등판 횟수)와 IP 합계 계산 ——
# Appearances: Name별 행 수
apps = df.groupby('Name').size().rename('Appearances')

# IP_sum: Name별 이닝 총합
ip_sum = df.groupby('Name')['IP'].sum().rename('IP')

# 두 시리즈를 하나로 합친 후 정리
summary = pd.concat([apps, ip_sum], axis=1).reset_index()

# —— 3) 필터링 —— 
valid = summary[
    (summary['Appearances'] >= 30) &
    (summary['IP']         >= 30)
]

# —— 4) 결과 출력 —— 
print(f"조건을 만족하는 투수: {len(valid)}명\n")
print(valid.sort_values(['IP','Appearances'], ascending=False).to_string(index=False))

print(df['Name'].nunique())

조건을 만족하는 투수: 160명

 Name  Appearances         IP
  박세웅          141 795.000000
  원태인          134 773.666667
  양현종          119 690.000000
  최원준          151 664.000000
  최원태          125 632.333333
  고영표          100 623.666667
  임찬규          122 620.666667
 쿠에바스          101 590.000000
  신민혁          127 548.333333
  임기영          179 536.333333
  오원석          129 530.000000
  백정현           97 520.666667
  김민우           99 515.000000
   반즈           86 507.333333
  김광현           89 504.000000
  한현희          159 479.333333
  엄상백           92 461.333333
  소형준           86 442.666667
  이재학           98 436.333333
  노경은          231 434.666667
  이영하          190 408.333333
  이의리           80 393.666667
  문승원          169 385.666667
  이태양          182 380.333333
  박종훈           77 374.666667
  후라도           60 374.000000
  나균안          111 367.333333
  최채흥           81 351.666667
  김민수          254 336.333333
  장시환          178 333.333333
  김재윤          305 324.000000
  서진용          315 31